# About this notebook
This notebook is for batching 2D analysis on multiple images at once.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
from glob import glob

import pandas as pd

from iob_ia.utils import filter_labels
from iob_ia.utils.io_utils import (
    gen_out_path,
    read_image,
    read_labels,
    save_image_channel,
    save_labels,
)
from iob_ia.utils.measure import overlapping_labels
from iob_ia.utils.segment import segment_2d_sub_projections
from iob_ia.utils.workflows3d import check_images

## 1. Define variables and check images

In [3]:
# Define the path to the folder (use '/', and make sure that the path also ends with '/')
path_to_folder = "G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2/"

# Image file type suffix
suffix = ".vsi"
# Channels that should be segmented (0-based indexing)
channels_to_segment = [1,2]
# In case metadata is missing (will be checked later...), specify the voxel size here
voxel_size = None  # or somthing like: (1.46, 0.65, 0.65)
# Range for substack projections before 2D segmentation
substack_size = 40.0

# workflow identifier, used for saving images (no need to change that)
wf = "IOB-2D-workflow_"

# Filtering: a list of dictionaries (same order as channels_to_segment)
filters = [
    # First channel
    {
        "projected_area": [42, 233],
        "intensity_mean": ["mean/3", float("inf")],
        #"intensity_mean": [250, float("inf")],
    },
    # Second channel
    {
        "projected_area": [42, 233],
        "intensity_mean": [150, float("inf")],
        "intensity_max": [None, 15000],
    }
]

*Info* - Filter dictionaires support dynamic values:

Instead of putting [min, max] absolute values for a certain measurement, you can define dynamic calculation based on the measurements itself.
Supported are:
- bimodal (same function as "estimate_threshold")
- mean
- median

Further you can adjust the dynamic thresholds by multiplication or division. E.g. by defining dictionaries as:
```
    {
        "area": ["median/4", "mean*5.5"],
        "intensity_mean": ["bimodal/2", float("inf")],
    }
```

## 2. Check images
Here we check if the images have correct and same metadata, as well as, whether the channels for segmentation are available in all images.


<font color='red'>WARNING: For testing purposes I 'manually' respecified to only use the first image (twice) for the analysis.</font>

<font color='red'>Please remove the 2nd code-line in the cell below.</font>

In [4]:
# check the image paths
image_paths = glob(path_to_folder + "*" + suffix)
image_paths = [image_paths[0], image_paths[0]]  # <-- This line needs to be removed
for p in image_paths:
    print(p)
voxel_size = check_images(image_paths=image_paths, channels_to_segment=channels_to_segment, voxel_size=voxel_size)
print('Final voxel size:', voxel_size)

G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01.vsi
G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01.vsi
Using manually defined voxel size.
All good!
Final voxel size: (1.46, 0.65, 0.65)


## 3. Segment images

In [5]:
if voxel_size is None:
    raise RuntimeError("The voxel size does not seem to be defined, probably the image check did not succeed...")
# Note do not keep images in memory
cp_mask_paths = []
image_channel_paths = []
bioio_images = []
cur_count = 1
mod_voxel_size = None  # place holder for projected voxel size
for p in image_paths:
    mask_paths = []
    channel_paths = []
    # Read the image
    image = read_image(p)
    bioio_images.append(image)
    img = None  # but don't load the image data yet
    for ch in channels_to_segment:
        cp_mask_path = gen_out_path(p, wf + f'cp-mask_ch{ch}-{image.channel_names[ch]}')
        channel_path = gen_out_path(p, wf + f'image_ch{ch}-{image.channel_names[ch]}.ome')
        if os.path.exists(cp_mask_path):
            print('Already segmented:', p, 'Channel', ch, image.channel_names[ch])
            # Load the mask
            cp_mask = read_labels(cp_mask_path)
            # If the corresponding image channel does not exist, create it and check if the size matches the mask
            if not os.path.exists(channel_path):
                import numpy as np
                if img is None:
                    print("Loading image", p, "...")
                    img = image.get_data()
                n_slices = int(round(substack_size / voxel_size[0], 0))
                img_ch = []
                for z in range(0, img[ch].shape[0], n_slices):
                    substack = np.max(img[ch][z:z + n_slices, :, :], axis=0)
                    img_ch.append(substack)
                img_ch = np.asarray(img_ch)
            else:
                # Load the image channel
                image_channel = read_image(channel_path)
                img_ch = image_channel.get_data()
            # Check that the loaded image channel and mask have the same shape
            if cp_mask.shape != img_ch.shape:
                raise RuntimeError(
                    f'Mask shape ({cp_mask.shape}) and image channel shape ({img_ch.shape}) do not match. '
                    "Please delete mask and image channel images from previous runs:\n"
                    f'  - {cp_mask_path}\n  - {channel_path}\n  - and any other associated images...'
                )
            mask_paths.append(cp_mask_path)
            channel_paths.append(channel_path)
            # remember the modified voxel size
            if mod_voxel_size is None:
                mod_voxel_size = (substack_size, voxel_size[1], voxel_size[2])
            del cp_mask, img_ch
        else:
            # Segment - Load the image data
            if img is None:
                print("Loading image", p, "...")
                img = image.get_data()
            cp_mask, img_ch, mod_voxel_size = segment_2d_sub_projections(
                img[ch],
                voxel_size=voxel_size,
                substack_size=substack_size
            )
            # Save the label image
            save_labels(cp_mask, cp_mask_path)
            # Save the single (raw) channel
            save_image_channel(img_ch, channel_path, voxel_size=mod_voxel_size)
            # Add the paths to the lists
            mask_paths.append(cp_mask_path)
            channel_paths.append(channel_path)
            del cp_mask, img_ch
    del img
    cp_mask_paths.append(mask_paths)
    image_channel_paths.append(channel_paths)
    print(f'\n    --- Finished processing image {cur_count} out of {len(image_paths)}\n')
    cur_count += 1
print("Finished segmenting all images and channels.")

Loading image G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01.vsi ...


Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.11.13 
torch version:  	2.8.0+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2025-10-27 16:01:59,489 [INFO] WRITING LOG OUTPUT TO C:\Users\Microscopy\.cellpose\run.log
2025-10-27 16:01:59,490 [INFO] 
cellpose version: 	4.0.6 
platform:       	win32 
python version: 	3.11.13 
torch version:  	2.8.0+cu126
>>> GPU activated: True
2025-10-27 16:01:59,491 [INFO] ** TORCH CUDA version installed and working. **
2025-10-27 16:01:59,492 [INFO] >>>> using GPU (CUDA)
2025-10-27 16:02:00,953 [INFO] >>>> loading model C:\Users\Microscopy\.cellpose\models\cpsam
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_6

## 4. Filtering label masks

In [9]:
# Check that everythign is available
masks_ok = True
channels_ok = True
for i in range(len(cp_mask_paths)):
    if None in cp_mask_paths[i]:
        masks_ok = False
    if None in image_channel_paths[i]:
        channels_ok = False
if not masks_ok:
    raise RuntimeError(f'Cannot proceed!\nProblems with mask paths:\n{cp_mask_paths}')
if not channels_ok:
    raise RuntimeError(f'Cannot proceed!\nProblems with image channel paths:\n{image_channel_paths}')

# Create filtered masks (will re-analyse and overwrite existing masks)
filtered_mask_paths = []
filtered_properties = []
cur_count = 1
for i in range(len(cp_mask_paths)):
    filtered_paths = []
    filtered_props = []
    for ch in range(len(cp_mask_paths[i])):
        cp_mask_path = cp_mask_paths[i][ch]
        channel_path = image_channel_paths[i][ch]
        filtered_path = cp_mask_path.replace('cp-mask_ch', "filtered-mask_ch")

        # Load the image an mask
        cp_mask = read_labels(cp_mask_path)
        channel = read_image(channel_path).get_data()
        # Measure properties
        properties = filter_labels.measure_label_props(cp_mask, channel, voxel_size=mod_voxel_size)
        del channel
        # Filter the mask and save it
        filtered_mask, updated_properties = filter_labels.filter_labels_by_property(cp_mask, props_table=properties, props=filters[ch])
        save_labels(filtered_mask, filtered_path)
        filtered_paths.append(filtered_path)
        filtered_props.append(updated_properties)
        del filtered_mask, cp_mask
    filtered_mask_paths.append(filtered_paths)
    filtered_properties.append(filtered_props)
    print(f'\n    --- Finished processing image {cur_count} out of {len(image_paths)}\n')
    cur_count += 1

Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_filtered-mask_ch1-488 C.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_filtered-mask_ch2-640 C.tif

    --- Finished processing image 1 out of 2

Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_filtered-mask_ch1-488 C.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_filtered-mask_ch2-640 C.tif

    --- Finished processing image 2 out of 2



## 5. Create double positive objects masks

In [7]:
# Only done if there exactly 2 masks per image availalbe
dp_mask_paths = []
for i in range(len(filtered_mask_paths)):
    if len(filtered_mask_paths[i]) == 2:
        # Load the images
        mask_1 = read_labels(filtered_mask_paths[i][0])
        mask_2 = read_labels(filtered_mask_paths[i][1])
        # Create saving path for 2nd-channel pos 1st-channel objects
        dp_path_1 = f'mask_{bioio_images[i].channel_names[channels_to_segment[1]]}-pos-{bioio_images[i].channel_names[channels_to_segment[0]]}-objects'
        dp_path_1 = gen_out_path(image_paths[i], wf + dp_path_1)
        dp_mask_1 = overlapping_labels(mask_1, mask_2)
        save_labels(dp_mask_1, dp_path_1)
        # Create saving path for 1st-channel pos 2nd-channel objects
        dp_path_2 = f'mask_{bioio_images[i].channel_names[channels_to_segment[0]]}-pos-{bioio_images[i].channel_names[channels_to_segment[1]]}-objects'
        dp_path_2 = gen_out_path(image_paths[i], wf + dp_path_2)
        dp_mask_2 = overlapping_labels(mask_2, mask_1)
        save_labels(dp_mask_2, dp_path_2)
        dp_mask_paths.append([dp_path_1, dp_path_2])
        del mask_1, mask_2
        del dp_mask_1, dp_mask_2
    else:
        # If not 2 channels: append empty path list
        dp_mask_paths.append([])

Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_640 C-pos-488 C-objects.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_488 C-pos-640 C-objects.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_640 C-pos-488 C-objects.tif
Saved labels to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2\20250520_10x_Nq146um_405_488_640_01_IOB-2D-workflow_mask_488 C-pos-640 C-objects.tif


## 6. Count all objects and create result table

In [8]:
result_table = None
for i_img in range(len(cp_mask_paths)):
    # Create result table
    if result_table is None:
        result_table = {
            "Image-Path": image_paths,
            "Image-Name": [],
        }
        for i_ch in range(len(cp_mask_paths[i_img])):
            result_table[f'cp-count-ch{channels_to_segment[i_ch]}'] = []
        for i_ch in range(len(cp_mask_paths[i_img])):
            result_table[f'object-count-ch{channels_to_segment[i_ch]}'] = []
        if len(cp_mask_paths[i_img]) == 2:
            result_table[f'ch{channels_to_segment[1]}pos-ch{channels_to_segment[0]}objects'] = []
            result_table[f'ch{channels_to_segment[0]}pos-ch{channels_to_segment[1]}objects'] = []
    # Image file names
    result_table["Image-Name"].append(
        os.path.basename(image_paths[i_img])
    )

    print(f'Counting image {i_img + 1} out of {len(cp_mask_paths)} images...')
    # Mask object counts
    for i_ch in range(len(cp_mask_paths[i_img])):
        # Load masks
        cp_mask = read_labels(cp_mask_paths[i_img][i_ch])
        filtered_mask = read_labels(filtered_mask_paths[i_img][i_ch])
        # count objects
        result_table[f'cp-count-ch{channels_to_segment[i_ch]}'].append(
            filter_labels.count_unique_labels(cp_mask)
        )
        result_table[f'object-count-ch{channels_to_segment[i_ch]}'].append(
            filter_labels.count_unique_labels(filtered_mask)
        )
    # double positive counts (only if path array is not empty)
    if dp_mask_paths[i_img]:
        # Load mask
        dp_mask_1 = read_labels(dp_mask_paths[i_img][0])
        dp_mask_2 = read_labels(dp_mask_paths[i_img][1])
        result_table[f'ch{channels_to_segment[1]}pos-ch{channels_to_segment[0]}objects'].append(
            filter_labels.count_unique_labels(dp_mask_1)
        )
        result_table[f'ch{channels_to_segment[0]}pos-ch{channels_to_segment[1]}objects'].append(
            filter_labels.count_unique_labels(dp_mask_2)
        )

result_table = pd.DataFrame.from_dict(result_table)
# Save table
rt_path = path_to_folder + wf + 'batch_result_table.csv'
pd.DataFrame.to_csv(result_table, rt_path)
print("Saved result table to:", rt_path)
result_table

Counting image 1 out of 2 images...
Counting image 2 out of 2 images...
Saved result table to: G:/20241120_IOB_Magdalena/20250520_TestClearing/Organoid 2/IOB-2D-workflow_result_table.csv


,Image-Path,Image-Name,cp-count-ch1,cp-count-ch2,object-count-ch1,object-count-ch2,ch2pos-ch1objects,ch1pos-ch2objects
0,G:/20241120_IOB_Magdalena/20250520_TestClearin...,20250520_10x_Nq146um_405_488_640_01.vsi,6806,7801,6299,6788,2809,3101
1,G:/20241120_IOB_Magdalena/20250520_TestClearin...,20250520_10x_Nq146um_405_488_640_01.vsi,6806,7801,6299,6788,2809,3101
